# 習題：數學物件、Python 型別與 AI 中的數值表示

在數學中，我們經常接觸下列概念：

- 自然數（Natural Numbers）
- 整數（Integers）
- 有理數（Rational Numbers）
- 實數（Real Numbers）
- 複數（Complex Numbers）
- 向量（Vectors）
- 矩陣（Matrices）
- 集合（Sets）
- 命題的真假（Truth Values）

本習題將探討：

```text
數學物件
    ↓
Python 型別
    ↓
CPython 實作
    ↓
AI 中的數值表示
```

---

## 第一部分：Python 如何表示數學物件？

請查閱 Python 文件或自行測試，完成下表。

| 數學概念 | Python 型別 | 範例 |
|----------|------------|------|
| 自然數 |  int|5 |
| 整數 |int | -3 |
| 有理數 | fractions.Fraction | Fraction(3, 4) |
| 實數 | float | 3.14159 |
| 複數 | complex | 3 + 4j |
| 向量 | list / tuple / numpy.ndarray | [1, 2, 3] 或 np.array([1, 2, 3]) |
| 矩陣 | numpy.ndarray / 巢狀 list |np.array([[1, 2], [3, 4]] )|
| 集合 | set | {1, 2, 3} |
| 命題真假 | bool | True 或 False |

並回答下列問題：

1. 哪些是 Python 的內建型別（Built-in Types）？int、float、complex、list、tuple、set、bool
2. 哪些需要額外模組才能較完整地表示？有理數：fractions.Fraction（內建 fractions 模組）,向量與矩陣：numpy.ndarray（第三方 numpy 套件）
3. Python 的型別分類與數學中的分類是否完全一致？不完全一致：精度限度：數學有無限與連續的概念；Python受記憶體與浮點數精度限制（如 $0.1 + 0.2 \neq 0.3$）。包含關係：數學為集合包含（$\mathbb{N} \subset \mathbb{Z} \subset \mathbb{R}$）；Python 型別彼此獨立，int 與 float 是不同的底層物件。缺少型別：Python 沒有獨立的「自然數」型別，統一由 int 表示。

---

## 第二部分：數學概念與 Python 型別

數學上有下列包含關係：

$$
\mathbb N
\subseteq
\mathbb Z
\subseteq
\mathbb Q
\subseteq
\mathbb R
\subseteq
\mathbb C
$$

請回答：

1. Python 是否為上述每一個數系都提供獨立型別？否：未為每一數系提供獨立型別（如自然數與整數共用 int，且無內建有理數型別）。
2. 為什麼自然數與整數都使用 `int` 表示？簡化設計與記憶體優勢：自然數為整數子集，Python 3 的 int 支援任意精度，能以統一邏輯高效處理正負整數與零。
3. Python 為什麼提供 `complex` 型別？支援科學計算與工程：複數廣泛用於電路、訊號處理與物理模擬，原生支援 complex（以 j 為虛數單位）可簡化數學語法。
4. Python 為什麼沒有內建的有理數型別？效能考量與使用頻率：分數運算需頻繁進行分子分母約分（GCD），開銷極高；日常計算用 float 速度快且已夠用，需要精確分數時調用標準庫 fractions 即可。
5. 下列三個物件的型別有何不同？補全常見三物件型別比較（若指 5、5.0、5 + 0j）：

5 為 int（整數）

5.0 為 float（雙精度浮點數）

5 + 0j 為 complex（複數）

```python
1
1.0
1+0j
```

---

## 第三部分：CPython 如何實作這些型別？

除了知道 Python 使用哪些型別表示數學物件之外，

我們也希望了解：

> Python 物件在 CPython 中是如何實作的？

請查閱資料，完成下表。

| 數學概念 | Python 型別 | CPython 實作 | 主要特點 |
|----------|------------|-------------|----------|
| 自然數 | `int` | PyLongObject | Python 3 將整數統一處理，可表示任意大小的非負整數（無限精度，受限於記憶體）。 |
| 整數 | `int` | PyLongObject | 採用變長陣列（以 30-bit 或 15-bit 為位元組單位）儲存，動態擴充，支援任意精度的正負整數。 |
| 實數 | `float` | PyFloatObject| 底層直接封裝 C 語言的 double（雙精度浮點數），遵循 IEEE 754 標準（64-bit）。 |
| 複數 | `complex` | PyComplexObject| 包含兩個 double 成員（real 與 imag），分別儲存實部與虛部。|
| 集合 | `set` |PySetObject |基於雜湊表（Hash Table）實作，元素不重複且無序，提供 $O(1)$ 平均時間複雜度的尋找與去重。 |
| 命題真假 | `bool` |PyLongObject | bool 為 int 的子類別，CPython 全局僅存在兩個單例（Py_True 與 Py_False） |
| 串列 | `list` |PyListObject| 底層為連續記憶體空間的動態指標陣列（Dynamic Array），支援 $O(1)$ 隨機存取與尾端動態擴充。 |

---

## 第四部分：深入探討各種型別

請針對第三部分中的每一種型別回答：

1. 它是否為 Python 的內建型別？是
2. 對應的 CPython 物件結構名稱為何？PyListObject
3. 它如何儲存資料？在連續記憶體中儲存指向各 Python 物件的指標陣列（Pointer Array）。
4. 是否有固定大小限制？無固定長度限制（理論上限僅受限於系統記憶體與 64 位元整數最大值）。
5. 若資料規模持續增加，如何擴充儲存空間？採用過度分配（Over-allocation）策略；容量不足時自動申請約 1.125 倍的預留空間並複製指標，實現 O(1) 均攤時間複雜度。

---

## 第五部分：Python 整數與浮點數

執行下列程式：

```python
print(10**100)
print(10**1000)
```

回答：

1. 為什麼 Python 能表示如此大的整數？採用變長陣列（底層以多個 15-bit 或 30-bit 的小整數組合成大數）儲存位元，會隨數值增大動態擴充記憶體空間。
2. Python 的 `int` 是否具有固定的位元數限制？沒有固定的位元數（如 32-bit 或 64-bit）限制。
3. 什麼是 Arbitrary Precision（任意精度）？指數值的精確度不受硬體暫存器固定長度限制，只要記憶體足夠，就能精確表示任意大小的整數而不發生溢位（Overflow）。
4. CPython 使用什麼物件來實作 `int`？使用 PyLongObject 結構體。
5. Python 整數大小真正的限制是什麼？受限於系統可用記憶體（RAM）的大小（以及字串轉換時的預設長度限制 sys.set_int_max_str_digits）。

---

再執行：

```python
0.1 + 0.2
```

```python
0.1 + 0.2 == 0.3
```

```python
format(0.1, ".20f")
```

回答：

1. 為什麼結果與數學上的答案不同？電腦內部採用二進位儲存資料，許多十進位小數（如 $0.1$）無法精確轉換為有限位數的二進位小數，因而產生捨入誤差（Rounding Error）。
2. 浮點數是否能精確表示所有小數？不能。只有能表示為二進位分數（分母為2^n 的組合，如 0.5、0.25）的小數才能精確儲存，其餘皆為近似值。
3. 什麼是浮點數的指浮點數能儲存與維持的有效數字位數（有效位數越多，誤差越小；IEEE 754 雙精度約為 53 個二進位位元，相當於 15~17 位十進位有效數字）。精度（precision）？
4. 什麼是浮點數的範指浮點數能表示的最大值與最小值區間（由指數位決定，IEEE 754 雙精度約為 $10^{-308} \sim 10^{308}$）。圍（range）？
5. 精度與範圍有何不同？精度決定數字有多「精準」（有效位數多寡）。

範圍決定數字有多「巨大或微小」（極限數值大小）。

---

## 第六部分：AI 模型中的數值表示

人工智慧模型需要儲存大量參數（Parameters）與權重（Weights）。

請查閱資料並回答：

1. AI 模型中的權重通常使用什麼型別儲存？主要使用浮點數型別（如 FP16、BF16、FP32），以及量化後的低位元整數型別（如 INT8、INT4）。
2. 什麼是 FP64？雙精度浮點數（Double Precision，64 位元），包含 1 位符號位、11 位指數位與 52 位小數位。精度極高，但記憶體與計算開銷巨大，極少用於深度學習。
3. 什麼是 FP32？單精度浮點數（Single Precision，32 位元），包含 1 位符號位、8 位指數位與 23 位小數位。過去是深度學習訓練與推理的標準格式。
4. 什麼是 FP16？半精度浮點數（Half Precision，16 位元），包含 1 位符號位、5 位指數位與 10 位小數位。記憶體用量僅 FP32 的一半且計算更快，但動態範圍較小，容易發生下溢（Underflow）。
5. 什麼是 BF16？Bfloat16（Brain Floating Point，16 位元），包含 1 位符號位、8 位指數位與 7 位小數位。它保留了與 FP32 相同的指數範圍（防止數值溢位），能大幅簡化混合精度訓練的穩定度。
6. 為什麼大型 AI 模型不直接使用 Python 的 `int` 作為權重？需要連續梯度計算：模型訓練仰賴微積分（梯度下降法），要求權重具備連續的小數調整能力，而 int 無法表示小數與微小變化。

硬體無法平行加速：Python 的 int（PyLongObject）是 CPU 上的高階動態物件，無法直接丟入 GPU/NPU 的 Tensor Core 等硬體加速器進行大規模平行陣列矩陣運算。

---

## 第七部分：比較不同數值表示方式

完成下表。

| 表示方式 | 位元數 | 是否有誤差 | 主要用途 |
|----------|--------|------------|----------|
| int |動態（變長）| 無誤差（精確整數） | 計數、索引、控制流程、精確整數運算以及模型量化（如 INT8/INT4 推理）。 |
| FP64 | 64-bit | 有（小數捨入誤差）| 高精度科學計算、物理模擬、天文書籍與金融精密運算。 |
| FP32 | 32-bit| 有（小數捨入誤差） | 通用單精度浮點運算、傳統 3D 圖形渲染、深度學習模型訓練的基準格式。 |
| FP16 | 16-bit | 有（小數捨入誤差） | 行動裝置與邊緣設備推理、混合精度（Mixed Precision）訓練以節省記憶體。 |
| BF16 | 16-bit |  有（小數捨入誤差）|大語言模型（LLM）與大規模 AI 模型的訓練與推理（提供與 FP32 相同的動態數值範圍）。 |

並回答：

1. 哪種表示方式最精確？int（Python 整數，任意精度無誤差）；若限定於浮點數型別則為 FP64（雙精度）。
2. 哪種表示方式最節省記憶體？FP16 與 BF16（皆僅需 16-bit / 2 Bytes）；若包含量化格式則為 INT4 或 INT8。
3. 哪種表示方式最常見於 AI 訓練？搭配 FP32 進行混合精度訓練）。
4. 哪種表示方式最常見於 FP16 以及量化後的 INT8 / INT4。AI 推論（Inference）？

---

## 第八部分：綜合思考

### 思考題一

Python 官方文件指出：

```python
int
```

具有 Unlimited Precision（無限制精度）。

請回答：

1. Unlimited Precision 是什麼意思？指數值表示與運算不受硬體暫存器固定位元數（如 32 或 64 位元）限制，無溢位（Overflow）風險。
2. 它是否代表可以儲存無限大的整數？理論上可以，但實務上受限於電腦的可用記憶體（RAM）大小。
3. CPython 如何實現這項特性？透過 PyLongObject 結構體採用變長陣列儲存數值，將大整數切分成多個 15-bit 或 30-bit 的區塊，並隨數值增大動態擴充記憶體。

---

### 思考題二

比較：

```python
int
```

與

```python
float
```

---

### 思考題三

比較下列兩種設計理念：

#### Python 的整數

```text
Arbitrary Precision
正確性優先
```

#### AI 模型常用的 FP16、BF16

```text
Limited Precision
效率優先
```
1. 為什麼 `int` 幾乎沒有固定上限？Python 3 的 int 採用動態記憶體分配（任意精度），數字變大時自動分配更多記憶體，上限僅受限於電腦的實體記憶體容量。
2. 為什麼 `float` 卻存在範圍限制？遵循 IEEE 754 標準，固定只佔用 64 位元（8 位元組），其中用於表示指數的位元有限，最大值約為 $1.79 \times 10^{308}$。
3. 為什麼 `float` 可能產生誤差？電腦底層使用二進位儲存，許多十進位小數（如 0.1）轉為二進位會變成無限循環小數，在有限的位元空間（52 位元尾數）截斷捨入後便產生了微小偏差。
請說明：

1. 兩種設計理念的差異。高精度（FP64/FP32）：追求極致精確度與廣大數值範圍，避免任何計算誤差。

低精度（FP16/BF16/INT8）：犧牲部分非必要的精確度，換取極高的計算速度、極低的記憶體佔用與傳輸頻寬。
2. 各自的優點與缺點。高精度：

優點：數值極度精準、極少下溢/溢位、訓練極穩定。

缺點：記憶體與頻寬需求極高、顯存易爆掉、硬體計算速度慢且極耗電。

低精度：

優點：省記憶體（顯存需求倍減）、計算吞吐量大（加速數倍）、能放進更多平行核心（Tensor Cores）。

缺點：精度較低、可能產生捨入誤差、部分格式易發生數值溢位或下溢（需要技巧穩定梯度）。
3. 為什麼大型語言模型通常選擇較低精度的表示方式？參數與記憶體瓶頸：千億參數模型若用 FP32 光載入權重就要數百 GB 顯存，低精度才能在有限 GPU 顯存中容納並運行模型。

深度學習對微小誤差不敏感：AI 權重與梯度調整本質上是機率與統計擬合，適度犧牲一點小數位精度完全不影響最終輸出的品質與效果。
4. 如果所有 AI 模型都改用 FP64，可能會產生哪些問題？
硬體成本與顯存暴增：顯存用量直接翻倍或翻四倍，現有 GPU 記憶體將嚴重不足。

運算速度極慢：通用 GPU 的 FP64 Tensor Core 運算能力遠低於 FP16/BF16，訓練與推論時間將延長數倍。

能耗與碳排放巨幅飆升：電力需求與散熱成本大增，大幅提升 AI 部署與營運成本。
---

### 挑戰題

請用自己的話解釋：

```python
x = 10**1000
```

為什麼 Python 可以正常運作；

而大型 AI 模型卻通常選擇使用：

```text
FP32
FP16
BF16
```

而不是任意精度整數或高精度浮點數？

請從：

- 記憶體使用量
- 計算速度
- 數值精度
- GPU 運算效率

等角度進行討論。